# Additional Experiment: full-param space in a stream setup

Here, I am going to visualize the results of `experiments\conf\experiment\full_param_space_stream.yaml`, where we try to isolate the dynamics of learning the full parameter space in a stream setup (52 tasks, same settings as the realsitic stream), from the partial-parameter space coverage of the realistic streams that we propose.

In [ ]:
from pathlib import Path

from skald import combine_runs
import seaborn as sns
import polars as pl

import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator
import numpy as np

from experiments.visualizations import set_paper_style

set_paper_style()

In [ ]:
metrics = combine_runs(
    directory=Path(r"W:\EndlessWM\experiments\reports\full_param_space_stream"),
    include_params=["method.name", "data.name", "data.dataset_seed", "seed"],
).filter(pl.col("method.name") != "joint", stream="IV", name="mAP")
metrics

In [ ]:
set_paper_style()

# Static complexity results for generator IV around 52 * 200 = 10,400 samples
# (7,500 samples -> 0.9616, 10,000 samples -> 0.9696, 25,000 samples -> 0.9852)
_static_iv_samples = np.array([250, 500, 750, 1000, 2500, 5000, 7500, 10000, 25000, 50000, 75000, 100000])
_static_iv_map = np.array([0.7496, 0.8158, 0.8439, 0.8675, 0.9196, 0.9483, 0.9616, 0.9696, 0.9852, 0.9900, 0.9932, 0.9937])
_static_iv_10400 = float(np.interp(np.log10(52 * 200), np.log10(_static_iv_samples), _static_iv_map))

# Edit legend labels and subplot titles here:
legend_labels = {
    "iv-f": "iv-f",
    "iv-r": "iv-r",
    "static_iv": r"static training on $52 \times 200$ samples from iv",
}
subplot_titles = {
    "finetuning": "a) finetuning",
    "replay": "b) replay",
}

_method_order = ["finetuning", "replay"]
_hue_order = [legend_labels["iv-f"], legend_labels["iv-r"]]

_plot_df = metrics.filter(
    pl.col("epoch").is_null(),
    pl.col("pattern") == "*",
    pl.col("method.name").is_in(_method_order),
).with_columns(pl.col("data.name").replace(legend_labels).alias("stream_type"))

fig = sns.relplot(
    data=_plot_df,
    x="task",
    y="value",
    hue="stream_type",
    hue_order=_hue_order,
    col="method.name",
    col_order=_method_order,
    kind="line",
    height=2.45,
    aspect=1.25,
    facet_kws={"sharex": False, "sharey": True},
    legend="full",
)

fig.set_axis_labels("task $t$", "mAP")
fig.set_titles("")

for _method, _ax in zip(_method_order, fig.axes.flat):
    _ax.axhline(
        _static_iv_10400,
        linestyle=":",
        color="0.2",
        linewidth=1.0,
        label=legend_labels["static_iv"],
    )
    _ax.set_ylim(0.25, 1.0)
    _ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    _ax.grid(True, axis="y", alpha=0.22, linewidth=0.5)
    _ax.set_xlabel(f"task $t$\n\n{subplot_titles[_method]}")

_handles, _labels = fig.axes.flat[0].get_legend_handles_labels()
if fig._legend is not None:
    fig._legend.remove()

fig.fig.legend(
    handles=_handles,
    labels=_labels,
    loc="lower center",
    bbox_to_anchor=(0.53, 0.96),
    ncol=len(_labels),
    title=None,
    frameon=True,
    borderaxespad=1.0,
    borderpad=0.25,
    handlelength=1.0,
    handletextpad=0.5,
    columnspacing=1.0,
    labelspacing=0.2,
)

fig.fig.tight_layout()

_output_dir = Path("../../reports/full_param_space_stream")
_output_dir.mkdir(parents=True, exist_ok=True)

for _stem in ["full_param_space_stream", "finetuning", "replay"]:
    fig.fig.savefig(_output_dir / f"{_stem}.png", bbox_inches="tight", dpi=300)
    fig.fig.savefig(_output_dir / f"{_stem}.pdf", bbox_inches="tight")

fig.fig